In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)

In [ ]:
# ============================================================
# CAE environment imports
# PyTorch was imported first intentionally
# ============================================================

import os

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"

from pathlib import Path
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

print("Environment imports completed.")

In [ ]:
# ============================================================
# Paper 1 — Baseline Models
# Cell 1: Imports, paths, configuration, and reproducibility
# ============================================================

# ------------------------------------------------------------
# Thread control — must be set BEFORE NumPy / scikit-learn
# ------------------------------------------------------------

import os

os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"

# ------------------------------------------------------------
# Core imports
# ------------------------------------------------------------

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

from threadpoolctl import threadpool_limits

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

PATCH_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "patches"
)

BASELINE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "baselines"
)

PCA_DIR = (
    BASELINE_DIR /
    "PCA_KMeans"
)

FIGURE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "figures"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "tables"
)

# ------------------------------------------------------------
# Create folders if needed
# ------------------------------------------------------------

for folder in [
    BASELINE_DIR,
    PCA_DIR,
    FIGURE_DIR,
    TABLE_DIR
]:

    folder.mkdir(
        parents=True,
        exist_ok=True
    )

# ------------------------------------------------------------
# Frozen experiment configuration
# ------------------------------------------------------------

PATCH_SIZES = [9, 13]

N_FOLDS = 5

N_CHANNELS = 14

# Candidate number of geophysical domains
K_VALUES = list(
    range(2, 11)
)

# Stability candidates identified from initial sweep
STABILITY_K_VALUES = [
    2,
    3,
    4
]

N_STABILITY_RUNS = 50

# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("Baseline experiment configuration")
print("-" * 60)

print(
    "Project root :",
    PROJECT_ROOT
)

print(
    "Patch sizes  :",
    PATCH_SIZES
)

print(
    "CV folds     :",
    N_FOLDS
)

print(
    "Channels     :",
    N_CHANNELS
)

print(
    "k sweep      :",
    K_VALUES
)

print(
    "Stability k  :",
    STABILITY_K_VALUES
)

print(
    "Stability runs:",
    N_STABILITY_RUNS
)

print(
    "Random seed  :",
    RANDOM_STATE
)

print(
    "OMP threads  :",
    os.environ["OMP_NUM_THREADS"]
)

print(
    "MKL threads  :",
    os.environ["MKL_NUM_THREADS"]
)

print("\nCell 1 completed successfully.")

In [ ]:
# ============================================================
# Diagnostic: inspect numerical thread libraries
# ============================================================

from threadpoolctl import threadpool_info
import pandas as pd

thread_info = threadpool_info()

thread_df = pd.DataFrame(thread_info)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

display(thread_df)

print("\nNumber of numerical thread pools detected:", len(thread_info))

In [ ]:
# ============================================================
# Cell 2: Load one patch dataset and inspect PCA input geometry
# ============================================================

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

fold_dir = (
    PATCH_DIR /
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}" /
    f"Fold_{TEST_FOLD}"
)

train_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

val_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_VALIDATION_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

train_meta_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_metadata.csv"
)

val_meta_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_VALIDATION_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_metadata.csv"
)

X_train_patch = np.load(train_file)
X_val_patch = np.load(val_file)

train_meta = pd.read_csv(train_meta_file)
val_meta = pd.read_csv(val_meta_file)

print("TRAIN patch tensor:")
print(X_train_patch.shape)

print("\nVALIDATION patch tensor:")
print(X_val_patch.shape)

# ------------------------------------------------------------
# Flatten patches for PCA baseline
# ------------------------------------------------------------

X_train_flat = X_train_patch.reshape(
    X_train_patch.shape[0],
    -1
)

X_val_flat = X_val_patch.reshape(
    X_val_patch.shape[0],
    -1
)

print("\nFlattened TRAIN matrix:")
print(X_train_flat.shape)

print("\nFlattened VALIDATION matrix:")
print(X_val_flat.shape)

print(
    "\nNumber of features per patch:",
    X_train_flat.shape[1]
)

assert X_train_flat.shape[1] == (
    TEST_PATCH_SIZE *
    TEST_PATCH_SIZE *
    N_CHANNELS
)

assert len(train_meta) == len(X_train_flat)
assert len(val_meta) == len(X_val_flat)

print(
    "\nIntegrity check passed: "
    "patch tensors and metadata are aligned."
)

print("\nCell 2 completed.")

In [ ]:
# ============================================================
# Cell 3: PCA fit on TRAIN only
#         Retain the minimum number of components explaining 95%
#         of the training variance
# ============================================================

# ------------------------------------------------------------
# Fit full PCA on TRAIN only
# ------------------------------------------------------------

pca_full = PCA(
    svd_solver="full",
    random_state=RANDOM_STATE
)

pca_full.fit(X_train_flat)

# ------------------------------------------------------------
# Cumulative explained variance
# ------------------------------------------------------------

explained_ratio = pca_full.explained_variance_ratio_

cumulative_variance = np.cumsum(
    explained_ratio
)

# Smallest number of PCs reaching 95% variance
n_components_95 = (
    np.searchsorted(
        cumulative_variance,
        0.95
    )
    + 1
)

print("Total possible PCA components:")
print(len(explained_ratio))

print("\nComponents required for 95% variance:")
print(n_components_95)

print(
    "\nCumulative explained variance at selected component:"
)
print(
    round(
        cumulative_variance[n_components_95 - 1],
        4
    )
)

# ------------------------------------------------------------
# Show variance at several reference levels
# ------------------------------------------------------------

for target in [0.80, 0.90, 0.95, 0.99]:

    n_required = (
        np.searchsorted(
            cumulative_variance,
            target
        )
        + 1
    )

    print(
        f"{target * 100:.0f}% variance -> "
        f"{n_required} PCs"
    )

# ------------------------------------------------------------
# Plot cumulative explained variance
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.plot(
    np.arange(
        1,
        len(cumulative_variance) + 1
    ),
    cumulative_variance,
    linewidth=1.5
)

ax.axhline(
    y=0.95,
    linestyle="--",
    linewidth=1.2,
    label="95% variance"
)

ax.axvline(
    x=n_components_95,
    linestyle=":",
    linewidth=1.2,
    label=f"{n_components_95} PCs"
)

ax.set_xlabel(
    "Number of Principal Components"
)

ax.set_ylabel(
    "Cumulative Explained Variance"
)

ax.set_title(
    "PCA Baseline — Fold 3, 13×13 Training Patches"
)

ax.set_ylim(
    0,
    1.02
)

ax.legend()

plt.tight_layout()

pca_variance_fig = (
    FIGURE_DIR /
    "Paper1_PCA_Fold3_13x13_explained_variance.png"
)

plt.savefig(
    pca_variance_fig,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nFigure saved to:")
print(pca_variance_fig)

print("\nCell 3 completed.")

In [ ]:
# ============================================================
# Cell 4: PCA(95%) + k-means sweep for Fold 3, 13x13
# ============================================================

# ------------------------------------------------------------
# Fit reduced PCA on TRAIN only
# ------------------------------------------------------------

pca_95 = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=RANDOM_STATE
)

Z_train = pca_95.fit_transform(
    X_train_flat
)

Z_val = pca_95.transform(
    X_val_flat
)

print("Reduced TRAIN representation:")
print(Z_train.shape)

print("\nReduced VALIDATION representation:")
print(Z_val.shape)

print(
    "\nRetained variance:",
    round(
        pca_95.explained_variance_ratio_.sum(),
        4
    )
)

# ------------------------------------------------------------
# Sweep k = 2 ... 10
# ------------------------------------------------------------

kmeans_records = []

kmeans_models = {}

for k in K_VALUES:

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=50
    )

    train_labels = model.fit_predict(
        Z_train
    )

    # --------------------------------------------------------
    # Internal clustering metrics on TRAIN
    # --------------------------------------------------------

    silhouette = silhouette_score(
        Z_train,
        train_labels
    )

    dbi = davies_bouldin_score(
        Z_train,
        train_labels
    )

    ch = calinski_harabasz_score(
        Z_train,
        train_labels
    )

    inertia = model.inertia_

    # --------------------------------------------------------
    # Assign VALIDATION samples to trained centroids
    # --------------------------------------------------------

    val_labels = model.predict(
        Z_val
    )

    # Number of validation samples in each cluster
    val_cluster_counts = np.bincount(
        val_labels,
        minlength=k
    )

    min_val_cluster = int(
        val_cluster_counts.min()
    )

    max_val_cluster = int(
        val_cluster_counts.max()
    )

    kmeans_models[k] = {
        "model": model,
        "train_labels": train_labels,
        "val_labels": val_labels
    }

    kmeans_records.append({
        "k": k,
        "Silhouette": silhouette,
        "Davies_Bouldin": dbi,
        "Calinski_Harabasz": ch,
        "Inertia": inertia,
        "Min_validation_cluster_size": min_val_cluster,
        "Max_validation_cluster_size": max_val_cluster
    })

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

kmeans_sweep_df = pd.DataFrame(
    kmeans_records
)

print("\nPCA + k-means sweep:")
display(
    kmeans_sweep_df.round(4)
)

# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

sweep_output = (
    PCA_DIR /
    "Fold3_13x13_PCA95_KMeans_k2to10.csv"
)

kmeans_sweep_df.to_csv(
    sweep_output,
    index=False
)

print("\nSweep table saved to:")
print(sweep_output)

print("\nCell 4 completed.")

In [ ]:
# ============================================================
# Cell 5: Visual comparison of PCA + k-means diagnostics
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(12, 9)
)

# ------------------------------------------------------------
# Silhouette
# ------------------------------------------------------------

axes[0, 0].plot(
    kmeans_sweep_df["k"],
    kmeans_sweep_df["Silhouette"],
    marker="o"
)

axes[0, 0].set_xlabel("Number of clusters (k)")
axes[0, 0].set_ylabel("Silhouette score")
axes[0, 0].set_title(
    "Silhouette — Higher is Better"
)

# ------------------------------------------------------------
# Davies-Bouldin
# ------------------------------------------------------------

axes[0, 1].plot(
    kmeans_sweep_df["k"],
    kmeans_sweep_df["Davies_Bouldin"],
    marker="o"
)

axes[0, 1].set_xlabel("Number of clusters (k)")
axes[0, 1].set_ylabel("Davies–Bouldin index")
axes[0, 1].set_title(
    "Davies–Bouldin — Lower is Better"
)

# ------------------------------------------------------------
# Calinski-Harabasz
# ------------------------------------------------------------

axes[1, 0].plot(
    kmeans_sweep_df["k"],
    kmeans_sweep_df["Calinski_Harabasz"],
    marker="o"
)

axes[1, 0].set_xlabel("Number of clusters (k)")
axes[1, 0].set_ylabel("Calinski–Harabasz index")
axes[1, 0].set_title(
    "Calinski–Harabasz — Higher is Better"
)

# ------------------------------------------------------------
# Inertia
# ------------------------------------------------------------

axes[1, 1].plot(
    kmeans_sweep_df["k"],
    kmeans_sweep_df["Inertia"],
    marker="o"
)

axes[1, 1].set_xlabel("Number of clusters (k)")
axes[1, 1].set_ylabel("Within-cluster inertia")
axes[1, 1].set_title(
    "K-Means Inertia"
)

fig.suptitle(
    "PCA + K-Means Baseline — Fold 3, 13×13",
    fontsize=15
)

plt.tight_layout()

metric_fig_output = (
    FIGURE_DIR /
    "Paper1_PCA_Fold3_13x13_KMeans_metrics.png"
)

plt.savefig(
    metric_fig_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Metric figure saved to:")
print(metric_fig_output)

print("\nCell 5 completed.")

In [ ]:
# ============================================================
# Cell 6: Repeated-seed clustering stability
#         PCA + KMeans for k = 2, 3, 4
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

from itertools import combinations
from threadpoolctl import threadpool_limits

# ------------------------------------------------------------
# Candidate cluster numbers from Cell 4
# ------------------------------------------------------------

STABILITY_K_VALUES = [2, 3, 4]

N_STABILITY_RUNS = 50

stability_records = []
stability_labels = {}

# ------------------------------------------------------------
# Repeated clustering
# ------------------------------------------------------------

for k in STABILITY_K_VALUES:

    print("=" * 70)
    print(f"STABILITY TEST: k = {k}")
    print("=" * 70)

    run_labels = []

    # Each seed represents an independent clustering run
    for seed in range(N_STABILITY_RUNS):

        model = KMeans(
            n_clusters=k,
            random_state=seed,
            n_init=1,
            algorithm="lloyd"
        )

        # Limit numerical-library thread usage
        with threadpool_limits(limits=2):

            labels = model.fit_predict(
                Z_train
            )

        run_labels.append(labels)

    # --------------------------------------------------------
    # Compare every pair of independent runs
    # --------------------------------------------------------

    ari_values = []
    nmi_values = []

    for i, j in combinations(
        range(N_STABILITY_RUNS),
        2
    ):

        ari = adjusted_rand_score(
            run_labels[i],
            run_labels[j]
        )

        nmi = normalized_mutual_info_score(
            run_labels[i],
            run_labels[j]
        )

        ari_values.append(ari)
        nmi_values.append(nmi)

    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    stability_records.append({
        "k": k,

        "Runs": N_STABILITY_RUNS,

        "ARI_mean": np.mean(ari_values),
        "ARI_std": np.std(
            ari_values,
            ddof=1
        ),
        "ARI_min": np.min(ari_values),
        "ARI_median": np.median(ari_values),

        "NMI_mean": np.mean(nmi_values),
        "NMI_std": np.std(
            nmi_values,
            ddof=1
        ),
        "NMI_min": np.min(nmi_values),
        "NMI_median": np.median(nmi_values)
    })

    stability_labels[k] = run_labels

    print(
        f"Mean ARI : "
        f"{np.mean(ari_values):.4f}"
    )

    print(
        f"Median ARI: "
        f"{np.median(ari_values):.4f}"
    )

    print(
        f"Mean NMI : "
        f"{np.mean(nmi_values):.4f}"
    )

    print(
        f"Median NMI: "
        f"{np.median(nmi_values):.4f}"
    )

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

stability_df = pd.DataFrame(
    stability_records
)

print("\nPCA + KMeans stability summary:")

display(
    stability_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

stability_output = (
    PCA_DIR /
    "Fold3_13x13_PCA95_KMeans_stability_k2to4.csv"
)

stability_df.to_csv(
    stability_output,
    index=False
)

print("\nStability table saved to:")
print(stability_output)

print("\nCell 6 completed.")

In [ ]:
# ============================================================
# Cell 7: Full five-fold PCA(95%) + KMeans(k=3) baseline
#         for 9x9 and 13x13 patches
# ============================================================

FINAL_PCA_K = 3

full_pca_records = []

for patch_size in PATCH_SIZES:

    print("\n" + "=" * 80)
    print(f"PATCH SIZE: {patch_size} x {patch_size}")
    print("=" * 80)

    for fold in range(1, N_FOLDS + 1):

        print(
            f"\nProcessing patch {patch_size}x{patch_size}, "
            f"Fold {fold}"
        )

        # ----------------------------------------------------
        # File paths
        # ----------------------------------------------------

        fold_dir = (
            PATCH_DIR /
            f"{patch_size}x{patch_size}" /
            f"Fold_{fold}"
        )

        train_file = (
            fold_dir /
            f"Fold{fold}_TRAIN_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        val_file = (
            fold_dir /
            f"Fold{fold}_VALIDATION_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        # ----------------------------------------------------
        # Load patches
        # ----------------------------------------------------

        X_train_patch = np.load(train_file)
        X_val_patch = np.load(val_file)

        # Flatten
        X_train_flat = X_train_patch.reshape(
            X_train_patch.shape[0],
            -1
        )

        X_val_flat = X_val_patch.reshape(
            X_val_patch.shape[0],
            -1
        )

        # ----------------------------------------------------
        # PCA fit on TRAIN only
        # ----------------------------------------------------

        pca = PCA(
            n_components=0.95,
            svd_solver="full",
            random_state=RANDOM_STATE
        )

        Z_train = pca.fit_transform(
            X_train_flat
        )

        Z_val = pca.transform(
            X_val_flat
        )

        n_pcs = Z_train.shape[1]

        retained_variance = (
            pca.explained_variance_ratio_.sum()
        )

        # ----------------------------------------------------
        # KMeans(k=3)
        # ----------------------------------------------------

        model = KMeans(
            n_clusters=FINAL_PCA_K,
            random_state=RANDOM_STATE,
            n_init=50,
            algorithm="lloyd"
        )

        with threadpool_limits(limits=2):

            train_labels = model.fit_predict(
                Z_train
            )

            val_labels = model.predict(
                Z_val
            )

        # ----------------------------------------------------
        # TRAIN clustering metrics
        # ----------------------------------------------------

        silhouette = silhouette_score(
            Z_train,
            train_labels
        )

        dbi = davies_bouldin_score(
            Z_train,
            train_labels
        )

        ch = calinski_harabasz_score(
            Z_train,
            train_labels
        )

        # ----------------------------------------------------
        # Cluster-size diagnostics
        # ----------------------------------------------------

        train_counts = np.bincount(
            train_labels,
            minlength=FINAL_PCA_K
        )

        val_counts = np.bincount(
            val_labels,
            minlength=FINAL_PCA_K
        )

        full_pca_records.append({
            "Patch_size": patch_size,
            "Validation_Fold": fold,

            "TRAIN_patches": len(X_train_patch),
            "VALIDATION_patches": len(X_val_patch),

            "PCA_components_95": n_pcs,
            "Retained_variance": retained_variance,

            "Silhouette": silhouette,
            "Davies_Bouldin": dbi,
            "Calinski_Harabasz": ch,

            "TRAIN_cluster_min": int(
                train_counts.min()
            ),
            "TRAIN_cluster_max": int(
                train_counts.max()
            ),

            "VALIDATION_cluster_min": int(
                val_counts.min()
            ),
            "VALIDATION_cluster_max": int(
                val_counts.max()
            )
        })

        print(
            f"PCs={n_pcs:3d} | "
            f"Sil={silhouette:.3f} | "
            f"DB={dbi:.3f} | "
            f"CH={ch:.1f} | "
            f"VAL min cluster={val_counts.min()}"
        )

# ------------------------------------------------------------
# Results dataframe
# ------------------------------------------------------------

full_pca_df = pd.DataFrame(
    full_pca_records
)

print("\n" + "=" * 80)
print("FULL PCA + KMEANS BASELINE RESULTS")
print("=" * 80)

display(
    full_pca_df.round(4)
)

# ------------------------------------------------------------
# Patch-size summary
# ------------------------------------------------------------

pca_patch_summary = (
    full_pca_df
    .groupby("Patch_size")
    .agg(
        mean_PCs=(
            "PCA_components_95",
            "mean"
        ),
        mean_Silhouette=(
            "Silhouette",
            "mean"
        ),
        std_Silhouette=(
            "Silhouette",
            "std"
        ),
        mean_Davies_Bouldin=(
            "Davies_Bouldin",
            "mean"
        ),
        mean_Calinski_Harabasz=(
            "Calinski_Harabasz",
            "mean"
        ),
        min_validation_cluster=(
            "VALIDATION_cluster_min",
            "min"
        )
    )
    .reset_index()
)

print("\nPATCH-SIZE SUMMARY")

display(
    pca_patch_summary.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

full_pca_output = (
    PCA_DIR /
    "Paper1_PCA95_KMeans_k3_full_5fold_results.csv"
)

summary_output = (
    PCA_DIR /
    "Paper1_PCA95_KMeans_k3_patchsize_summary.csv"
)

full_pca_df.to_csv(
    full_pca_output,
    index=False
)

pca_patch_summary.to_csv(
    summary_output,
    index=False
)

print("\nSaved:")
print(full_pca_output)
print(summary_output)

print("\nCell 7 completed.")

In [ ]:
# ============================================================
# Cell 8: NMF baseline setup
#         Fold 3, 13x13 representative test
# ============================================================

from sklearn.decomposition import NMF

# ------------------------------------------------------------
# Reuse Fold 3, 13x13 patch tensors
# ------------------------------------------------------------

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

fold_dir = (
    PATCH_DIR /
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}" /
    f"Fold_{TEST_FOLD}"
)

train_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

val_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_VALIDATION_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

X_train_patch = np.load(train_file)
X_val_patch = np.load(val_file)

# Flatten patches
X_train_flat_nmf = X_train_patch.reshape(
    X_train_patch.shape[0],
    -1
).astype(np.float64)

X_val_flat_nmf = X_val_patch.reshape(
    X_val_patch.shape[0],
    -1
).astype(np.float64)

print("TRAIN flat shape:", X_train_flat_nmf.shape)
print("VALIDATION flat shape:", X_val_flat_nmf.shape)

# ------------------------------------------------------------
# TRAIN-derived nonnegative shift
#
# NMF cannot accept negative values.
# For each flattened feature position:
#
# shift_j = max(0, -min(TRAIN_j))
#
# The SAME shift is then applied to validation.
# ------------------------------------------------------------

train_feature_min = np.min(
    X_train_flat_nmf,
    axis=0
)

feature_shift = np.maximum(
    0.0,
    -train_feature_min
)

X_train_nonneg = (
    X_train_flat_nmf +
    feature_shift
)

X_val_nonneg = (
    X_val_flat_nmf +
    feature_shift
)

# ------------------------------------------------------------
# Tiny validation negatives can occur if validation contains
# values below the TRAIN minimum.
#
# We must diagnose this before clipping.
# ------------------------------------------------------------

negative_val_mask = (
    X_val_nonneg < 0
)

n_negative_val = int(
    negative_val_mask.sum()
)

min_validation_shifted = float(
    X_val_nonneg.min()
)

print("\nAfter TRAIN-derived shift:")
print(
    "TRAIN minimum:",
    round(float(X_train_nonneg.min()), 8)
)

print(
    "VALIDATION minimum:",
    round(min_validation_shifted, 8)
)

print(
    "Negative validation entries:",
    n_negative_val
)

print(
    "Negative validation percentage:",
    round(
        100.0 *
        n_negative_val /
        X_val_nonneg.size,
        4
    ),
    "%"
)

# ------------------------------------------------------------
# Clip validation values below zero.
#
# This uses no validation statistics; it only enforces NMF's
# mathematical nonnegativity requirement after the shift was
# defined exclusively from training data.
# ------------------------------------------------------------

X_val_nonneg = np.clip(
    X_val_nonneg,
    0.0,
    None
)

X_train_nonneg = np.clip(
    X_train_nonneg,
    0.0,
    None
)

assert np.all(X_train_nonneg >= 0)
assert np.all(X_val_nonneg >= 0)

print(
    "\nNonnegativity check passed."
)

print("\nCell 8 completed.")

In [ ]:
# ============================================================
# Cell 9: Select NMF latent dimensionality
#         Fold 3, 13x13 representative test
# ============================================================

from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ------------------------------------------------------------
# Candidate NMF latent dimensions
# ------------------------------------------------------------

NMF_COMPONENTS = [
    10,
    20,
    30,
    50,
    75,
    100
]

TEMP_K = 3

nmf_rank_records = {}
nmf_rank_results = []

# Frobenius norm of the training data
train_norm = np.linalg.norm(
    X_train_nonneg,
    ord="fro"
)

for n_components in NMF_COMPONENTS:

    print("=" * 70)
    print(
        f"NMF COMPONENTS: {n_components}"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Fit NMF on TRAIN only
    # --------------------------------------------------------

    nmf = NMF(
        n_components=n_components,
        init="nndsvda",
        solver="cd",
        beta_loss="frobenius",
        max_iter=1000,
        tol=1e-4,
        random_state=RANDOM_STATE
    )

    W_train = nmf.fit_transform(
        X_train_nonneg
    )

    W_val = nmf.transform(
        X_val_nonneg
    )

    # --------------------------------------------------------
    # Normalized reconstruction error
    # --------------------------------------------------------

    reconstruction_error = (
        nmf.reconstruction_err_
    )

    normalized_reconstruction_error = (
        reconstruction_error /
        train_norm
    )

    # --------------------------------------------------------
    # Temporary k=3 clustering in NMF latent space
    # --------------------------------------------------------

    kmeans = KMeans(
        n_clusters=TEMP_K,
        random_state=RANDOM_STATE,
        n_init=50,
        algorithm="lloyd"
    )

    train_labels = kmeans.fit_predict(
        W_train
    )

    val_labels = kmeans.predict(
        W_val
    )

    silhouette = silhouette_score(
        W_train,
        train_labels
    )

    dbi = davies_bouldin_score(
        W_train,
        train_labels
    )

    ch = calinski_harabasz_score(
        W_train,
        train_labels
    )

    train_counts = np.bincount(
        train_labels,
        minlength=TEMP_K
    )

    val_counts = np.bincount(
        val_labels,
        minlength=TEMP_K
    )

    nmf_rank_results.append({
        "NMF_components": n_components,
        "Reconstruction_error": reconstruction_error,
        "Normalized_reconstruction_error":
            normalized_reconstruction_error,
        "Silhouette_k3": silhouette,
        "Davies_Bouldin_k3": dbi,
        "Calinski_Harabasz_k3": ch,
        "TRAIN_min_cluster": int(
            train_counts.min()
        ),
        "VALIDATION_min_cluster": int(
            val_counts.min()
        ),
        "NMF_iterations": nmf.n_iter_
    })

    nmf_rank_records[n_components] = {
        "model": nmf,
        "W_train": W_train,
        "W_val": W_val
    }

    print(
        f"Normalized reconstruction error: "
        f"{normalized_reconstruction_error:.4f}"
    )

    print(
        f"Silhouette (k=3): "
        f"{silhouette:.4f}"
    )

    print(
        f"Davies-Bouldin (k=3): "
        f"{dbi:.4f}"
    )

    print(
        f"Calinski-Harabasz (k=3): "
        f"{ch:.2f}"
    )

    print(
        f"Validation minimum cluster size: "
        f"{val_counts.min()}"
    )

    print(
        f"NMF iterations: "
        f"{nmf.n_iter_}"
    )


# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

nmf_rank_df = pd.DataFrame(
    nmf_rank_results
)

print("\nNMF LATENT-RANK SWEEP")
print("=" * 70)

display(
    nmf_rank_df.round(4)
)

# ------------------------------------------------------------
# Save table
# ------------------------------------------------------------

NMF_DIR = (
    BASELINE_DIR /
    "NMF_KMeans"
)

NMF_DIR.mkdir(
    parents=True,
    exist_ok=True
)

nmf_rank_output = (
    NMF_DIR /
    "Fold3_13x13_NMF_rank_sweep.csv"
)

nmf_rank_df.to_csv(
    nmf_rank_output,
    index=False
)

print("\nSaved to:")
print(nmf_rank_output)

print("\nCell 9 completed.")

In [ ]:
# ============================================================
# Cell 10: Confirm NMF rank-10 convergence
#          Fold 3, 13x13
# ============================================================

FINAL_NMF_COMPONENTS = 10

nmf_10 = NMF(
    n_components=FINAL_NMF_COMPONENTS,
    init="nndsvda",
    solver="cd",
    beta_loss="frobenius",
    max_iter=5000,
    tol=1e-5,
    random_state=RANDOM_STATE
)

W_train_10 = nmf_10.fit_transform(
    X_train_nonneg
)

W_val_10 = nmf_10.transform(
    X_val_nonneg
)

# ------------------------------------------------------------
# Reconstruction diagnostics
# ------------------------------------------------------------

train_norm = np.linalg.norm(
    X_train_nonneg,
    ord="fro"
)

normalized_reconstruction_error = (
    nmf_10.reconstruction_err_
    / train_norm
)

print("NMF rank:", FINAL_NMF_COMPONENTS)
print("Iterations used:", nmf_10.n_iter_)

print(
    "Normalized reconstruction error:",
    round(normalized_reconstruction_error, 6)
)

print(
    "TRAIN latent shape:",
    W_train_10.shape
)

print(
    "VALIDATION latent shape:",
    W_val_10.shape
)

# ------------------------------------------------------------
# Re-run k=3 diagnostic
# ------------------------------------------------------------

kmeans_nmf10 = KMeans(
    n_clusters=3,
    random_state=RANDOM_STATE,
    n_init=50,
    algorithm="lloyd"
)

train_labels_nmf10 = kmeans_nmf10.fit_predict(
    W_train_10
)

val_labels_nmf10 = kmeans_nmf10.predict(
    W_val_10
)

sil_nmf10 = silhouette_score(
    W_train_10,
    train_labels_nmf10
)

db_nmf10 = davies_bouldin_score(
    W_train_10,
    train_labels_nmf10
)

ch_nmf10 = calinski_harabasz_score(
    W_train_10,
    train_labels_nmf10
)

train_counts_nmf10 = np.bincount(
    train_labels_nmf10,
    minlength=3
)

val_counts_nmf10 = np.bincount(
    val_labels_nmf10,
    minlength=3
)

print("\nClustering diagnostics")
print("-" * 60)

print(
    "Silhouette:",
    round(sil_nmf10, 4)
)

print(
    "Davies-Bouldin:",
    round(db_nmf10, 4)
)

print(
    "Calinski-Harabasz:",
    round(ch_nmf10, 2)
)

print(
    "TRAIN cluster counts:",
    train_counts_nmf10
)

print(
    "VALIDATION cluster counts:",
    val_counts_nmf10
)

print(
    "Minimum TRAIN cluster:",
    train_counts_nmf10.min()
)

print(
    "Minimum VALIDATION cluster:",
    val_counts_nmf10.min()
)

print("\nCell 10 completed.")

In [ ]:
# ============================================================
# Cell 11: NMF(rank=10) + KMeans sweep
#          Fold 3, 13x13 representative test
# ============================================================

NMF_K_VALUES = list(range(2, 11))

nmf_k_records = []
nmf_k_models = {}

for k in NMF_K_VALUES:

    print("=" * 70)
    print(f"NMF rank 10 | KMEANS k = {k}")
    print("=" * 70)

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=50,
        algorithm="lloyd"
    )

    with threadpool_limits(limits=2):

        train_labels = model.fit_predict(
            W_train_10
        )

        val_labels = model.predict(
            W_val_10
        )

    # --------------------------------------------------------
    # Internal TRAIN metrics
    # --------------------------------------------------------

    silhouette = silhouette_score(
        W_train_10,
        train_labels
    )

    dbi = davies_bouldin_score(
        W_train_10,
        train_labels
    )

    ch = calinski_harabasz_score(
        W_train_10,
        train_labels
    )

    inertia = model.inertia_

    # --------------------------------------------------------
    # Cluster occupancy
    # --------------------------------------------------------

    train_counts = np.bincount(
        train_labels,
        minlength=k
    )

    val_counts = np.bincount(
        val_labels,
        minlength=k
    )

    nmf_k_records.append({
        "k": k,
        "Silhouette": silhouette,
        "Davies_Bouldin": dbi,
        "Calinski_Harabasz": ch,
        "Inertia": inertia,
        "TRAIN_min_cluster": int(
            train_counts.min()
        ),
        "TRAIN_max_cluster": int(
            train_counts.max()
        ),
        "VALIDATION_min_cluster": int(
            val_counts.min()
        ),
        "VALIDATION_max_cluster": int(
            val_counts.max()
        ),
        "Empty_validation_clusters": int(
            np.sum(val_counts == 0)
        )
    })

    nmf_k_models[k] = {
        "model": model,
        "train_labels": train_labels,
        "val_labels": val_labels
    }

    print(
        f"Silhouette       : {silhouette:.4f}"
    )

    print(
        f"Davies-Bouldin   : {dbi:.4f}"
    )

    print(
        f"Calinski-Harabasz: {ch:.2f}"
    )

    print(
        f"TRAIN counts     : {train_counts}"
    )

    print(
        f"VALIDATION counts: {val_counts}"
    )

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

nmf_k_sweep_df = pd.DataFrame(
    nmf_k_records
)

print("\nNMF rank-10 + KMeans sweep")
print("=" * 70)

display(
    nmf_k_sweep_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

nmf_k_sweep_output = (
    NMF_DIR /
    "Fold3_13x13_NMF10_KMeans_k2to10.csv"
)

nmf_k_sweep_df.to_csv(
    nmf_k_sweep_output,
    index=False
)

print("\nSaved to:")
print(nmf_k_sweep_output)

print("\nCell 11 completed.")

In [ ]:
# ============================================================
# Cell 12: NMF(rank=10) + KMeans(k=2) stability
#          Fold 3, 13x13
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

from itertools import combinations

FINAL_NMF_K = 2
N_STABILITY_RUNS = 50

nmf_stability_labels = []

for seed in range(N_STABILITY_RUNS):

    model = KMeans(
        n_clusters=FINAL_NMF_K,
        random_state=seed,
        n_init=1,
        algorithm="lloyd"
    )

    labels = model.fit_predict(
        W_train_10
    )

    nmf_stability_labels.append(
        labels
    )

# ------------------------------------------------------------
# Pairwise stability
# ------------------------------------------------------------

ari_values = []
nmi_values = []

for i, j in combinations(
    range(N_STABILITY_RUNS),
    2
):

    ari_values.append(
        adjusted_rand_score(
            nmf_stability_labels[i],
            nmf_stability_labels[j]
        )
    )

    nmi_values.append(
        normalized_mutual_info_score(
            nmf_stability_labels[i],
            nmf_stability_labels[j]
        )
    )

nmf_stability_summary = pd.DataFrame(
    [{
        "NMF_components": FINAL_NMF_COMPONENTS,
        "k": FINAL_NMF_K,
        "Runs": N_STABILITY_RUNS,

        "ARI_mean": np.mean(ari_values),
        "ARI_std": np.std(
            ari_values,
            ddof=1
        ),
        "ARI_min": np.min(ari_values),
        "ARI_median": np.median(ari_values),

        "NMI_mean": np.mean(nmi_values),
        "NMI_std": np.std(
            nmi_values,
            ddof=1
        ),
        "NMI_min": np.min(nmi_values),
        "NMI_median": np.median(nmi_values)
    }]
)

display(
    nmf_stability_summary.round(4)
)

nmf_stability_output = (
    NMF_DIR /
    "Fold3_13x13_NMF10_KMeans_k2_stability.csv"
)

nmf_stability_summary.to_csv(
    nmf_stability_output,
    index=False
)

print("\nSaved to:")
print(nmf_stability_output)

print("\nCell 12 completed.")

In [ ]:
# ============================================================
# Cell 13: Full five-fold NMF(rank=10) + KMeans(k=2) baseline
#          for 9x9 and 13x13 patches
# ============================================================

FINAL_NMF_COMPONENTS = 10
FINAL_NMF_K = 2

full_nmf_records = []

for patch_size in PATCH_SIZES:

    print("\n" + "=" * 80)
    print(f"PATCH SIZE: {patch_size} x {patch_size}")
    print("=" * 80)

    for fold in range(1, N_FOLDS + 1):

        print(
            f"\nProcessing patch {patch_size}x{patch_size}, "
            f"Fold {fold}"
        )

        # ----------------------------------------------------
        # Load fold-specific tensors
        # ----------------------------------------------------

        fold_dir = (
            PATCH_DIR /
            f"{patch_size}x{patch_size}" /
            f"Fold_{fold}"
        )

        train_file = (
            fold_dir /
            f"Fold{fold}_TRAIN_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        val_file = (
            fold_dir /
            f"Fold{fold}_VALIDATION_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        X_train_patch = np.load(train_file)
        X_val_patch = np.load(val_file)

        # ----------------------------------------------------
        # Flatten patches
        # ----------------------------------------------------

        X_train_flat = (
            X_train_patch
            .reshape(
                X_train_patch.shape[0],
                -1
            )
            .astype(np.float64)
        )

        X_val_flat = (
            X_val_patch
            .reshape(
                X_val_patch.shape[0],
                -1
            )
            .astype(np.float64)
        )

        # ----------------------------------------------------
        # TRAIN-derived nonnegative shift
        # ----------------------------------------------------

        train_feature_min = np.min(
            X_train_flat,
            axis=0
        )

        feature_shift = np.maximum(
            0.0,
            -train_feature_min
        )

        X_train_nonneg = (
            X_train_flat +
            feature_shift
        )

        X_val_nonneg_raw = (
            X_val_flat +
            feature_shift
        )

        # Diagnose validation values below TRAIN minimum
        negative_val_mask = (
            X_val_nonneg_raw < 0
        )

        n_negative_val = int(
            negative_val_mask.sum()
        )

        negative_val_pct = (
            100.0 *
            n_negative_val /
            X_val_nonneg_raw.size
        )

        # Enforce NMF nonnegativity
        X_train_nonneg = np.clip(
            X_train_nonneg,
            0.0,
            None
        )

        X_val_nonneg = np.clip(
            X_val_nonneg_raw,
            0.0,
            None
        )

        # ----------------------------------------------------
        # Fit NMF on TRAIN only
        # ----------------------------------------------------

        nmf = NMF(
            n_components=FINAL_NMF_COMPONENTS,
            init="nndsvda",
            solver="cd",
            beta_loss="frobenius",
            max_iter=5000,
            tol=1e-5,
            random_state=RANDOM_STATE
        )

        W_train = nmf.fit_transform(
            X_train_nonneg
        )

        W_val = nmf.transform(
            X_val_nonneg
        )

        # ----------------------------------------------------
        # Normalized reconstruction error
        # ----------------------------------------------------

        train_norm = np.linalg.norm(
            X_train_nonneg,
            ord="fro"
        )

        normalized_reconstruction_error = (
            nmf.reconstruction_err_
            / train_norm
        )

        # ----------------------------------------------------
        # KMeans(k=2)
        # ----------------------------------------------------

        model = KMeans(
            n_clusters=FINAL_NMF_K,
            random_state=RANDOM_STATE,
            n_init=50,
            algorithm="lloyd"
        )

        train_labels = model.fit_predict(
            W_train
        )

        val_labels = model.predict(
            W_val
        )

        # ----------------------------------------------------
        # TRAIN clustering metrics
        # ----------------------------------------------------

        silhouette = silhouette_score(
            W_train,
            train_labels
        )

        dbi = davies_bouldin_score(
            W_train,
            train_labels
        )

        ch = calinski_harabasz_score(
            W_train,
            train_labels
        )

        # ----------------------------------------------------
        # Occupancy diagnostics
        # ----------------------------------------------------

        train_counts = np.bincount(
            train_labels,
            minlength=FINAL_NMF_K
        )

        val_counts = np.bincount(
            val_labels,
            minlength=FINAL_NMF_K
        )

        full_nmf_records.append({
            "Patch_size": patch_size,
            "Validation_Fold": fold,

            "TRAIN_patches": len(X_train_patch),
            "VALIDATION_patches": len(X_val_patch),

            "NMF_components": FINAL_NMF_COMPONENTS,
            "NMF_iterations": nmf.n_iter_,

            "Normalized_reconstruction_error":
                normalized_reconstruction_error,

            "Negative_validation_entries":
                n_negative_val,

            "Negative_validation_percent":
                negative_val_pct,

            "Silhouette": silhouette,
            "Davies_Bouldin": dbi,
            "Calinski_Harabasz": ch,

            "TRAIN_cluster_min": int(
                train_counts.min()
            ),

            "TRAIN_cluster_max": int(
                train_counts.max()
            ),

            "VALIDATION_cluster_min": int(
                val_counts.min()
            ),

            "VALIDATION_cluster_max": int(
                val_counts.max()
            ),

            "Empty_validation_clusters": int(
                np.sum(val_counts == 0)
            )
        })

        print(
            f"Iter={nmf.n_iter_:4d} | "
            f"Recon={normalized_reconstruction_error:.3f} | "
            f"Sil={silhouette:.3f} | "
            f"DB={dbi:.3f} | "
            f"CH={ch:.1f} | "
            f"VAL min={val_counts.min()} | "
            f"neg={negative_val_pct:.2f}%"
        )

# ------------------------------------------------------------
# Full results table
# ------------------------------------------------------------

full_nmf_df = pd.DataFrame(
    full_nmf_records
)

print("\n" + "=" * 80)
print("FULL NMF + KMEANS BASELINE RESULTS")
print("=" * 80)

display(
    full_nmf_df.round(4)
)

# ------------------------------------------------------------
# Patch-size summary
# ------------------------------------------------------------

nmf_patch_summary = (
    full_nmf_df
    .groupby("Patch_size")
    .agg(
        mean_reconstruction_error=(
            "Normalized_reconstruction_error",
            "mean"
        ),
        mean_Silhouette=(
            "Silhouette",
            "mean"
        ),
        std_Silhouette=(
            "Silhouette",
            "std"
        ),
        mean_Davies_Bouldin=(
            "Davies_Bouldin",
            "mean"
        ),
        mean_Calinski_Harabasz=(
            "Calinski_Harabasz",
            "mean"
        ),
        min_validation_cluster=(
            "VALIDATION_cluster_min",
            "min"
        ),
        max_negative_validation_percent=(
            "Negative_validation_percent",
            "max"
        ),
        max_NMF_iterations=(
            "NMF_iterations",
            "max"
        )
    )
    .reset_index()
)

print("\nPATCH-SIZE SUMMARY")

display(
    nmf_patch_summary.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

full_nmf_output = (
    NMF_DIR /
    "Paper1_NMF10_KMeans_k2_full_5fold_results.csv"
)

nmf_summary_output = (
    NMF_DIR /
    "Paper1_NMF10_KMeans_k2_patchsize_summary.csv"
)

full_nmf_df.to_csv(
    full_nmf_output,
    index=False
)

nmf_patch_summary.to_csv(
    nmf_summary_output,
    index=False
)

print("\nSaved:")
print(full_nmf_output)
print(nmf_summary_output)

print("\nCell 13 completed.")

In [ ]:
# ============================================================
# Cell 14: Check / install MiniSom for SOM baseline
# ============================================================

try:
    from minisom import MiniSom
    print("MiniSom is already installed.")
except ImportError:
    print("MiniSom is not installed.")
    print("Install it in Anaconda Prompt with:")
    print("pip install minisom")

In [ ]:
# ============================================================
# Cell 15: Representative SOM baseline
#          Fold 3, 13x13
#          Self-contained version
# ============================================================

from minisom import MiniSom

# ------------------------------------------------------------
# Representative configuration
# ------------------------------------------------------------

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

# ------------------------------------------------------------
# Load Fold 3, 13x13 patch tensors
# ------------------------------------------------------------

fold_dir = (
    PATCH_DIR /
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}" /
    f"Fold_{TEST_FOLD}"
)

train_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

val_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_VALIDATION_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

X_train_patch_som = np.load(train_file)
X_val_patch_som = np.load(val_file)

print("TRAIN patch tensor:", X_train_patch_som.shape)
print("VALIDATION patch tensor:", X_val_patch_som.shape)

# ------------------------------------------------------------
# Flatten patches
# ------------------------------------------------------------

X_train_flat_som = X_train_patch_som.reshape(
    X_train_patch_som.shape[0],
    -1
)

X_val_flat_som = X_val_patch_som.reshape(
    X_val_patch_som.shape[0],
    -1
)

print("\nFlattened TRAIN:", X_train_flat_som.shape)
print("Flattened VALIDATION:", X_val_flat_som.shape)

# ------------------------------------------------------------
# PCA fitted on TRAIN only
# Retain 95% training variance
# ------------------------------------------------------------

som_pca = PCA(
    n_components=0.95,
    svd_solver="full",
    random_state=RANDOM_STATE
)

Z_train_som = som_pca.fit_transform(
    X_train_flat_som
)

Z_val_som = som_pca.transform(
    X_val_flat_som
)

print("\nSOM TRAIN latent shape:", Z_train_som.shape)
print("SOM VALIDATION latent shape:", Z_val_som.shape)

print(
    "Retained PCA variance:",
    round(
        som_pca.explained_variance_ratio_.sum(),
        4
    )
)

# ------------------------------------------------------------
# SOM map configuration
# ------------------------------------------------------------

SOM_X = 5
SOM_Y = 5

som = MiniSom(
    x=SOM_X,
    y=SOM_Y,
    input_len=Z_train_som.shape[1],
    sigma=1.0,
    learning_rate=0.5,
    neighborhood_function="gaussian",
    random_seed=RANDOM_STATE
)

# PCA initialization using TRAIN only
som.pca_weights_init(
    Z_train_som
)

# ------------------------------------------------------------
# Train SOM
# ------------------------------------------------------------

N_SOM_ITERATIONS = 5000

som.train_random(
    data=Z_train_som,
    num_iteration=N_SOM_ITERATIONS,
    verbose=False
)

print("\nSOM training completed.")

# ------------------------------------------------------------
# Quantization errors
# ------------------------------------------------------------

train_qe = som.quantization_error(
    Z_train_som
)

val_qe = som.quantization_error(
    Z_val_som
)

print(
    "TRAIN quantization error:",
    round(train_qe, 4)
)

print(
    "VALIDATION quantization error:",
    round(val_qe, 4)
)

# ------------------------------------------------------------
# Best Matching Units
# ------------------------------------------------------------

train_bmu = np.array(
    [
        som.winner(x)
        for x in Z_train_som
    ]
)

val_bmu = np.array(
    [
        som.winner(x)
        for x in Z_val_som
    ]
)

print("\nTRAIN BMU shape:", train_bmu.shape)
print("VALIDATION BMU shape:", val_bmu.shape)

# ------------------------------------------------------------
# Occupied neurons
# ------------------------------------------------------------

train_unique_bmu = np.unique(
    train_bmu,
    axis=0
)

val_unique_bmu = np.unique(
    val_bmu,
    axis=0
)

print(
    "Occupied TRAIN SOM neurons:",
    len(train_unique_bmu),
    "/",
    SOM_X * SOM_Y
)

print(
    "Occupied VALIDATION SOM neurons:",
    len(val_unique_bmu),
    "/",
    SOM_X * SOM_Y
)

# ------------------------------------------------------------
# U-Matrix
# ------------------------------------------------------------

u_matrix = som.distance_map()

fig, ax = plt.subplots(
    figsize=(7, 6)
)

im = ax.imshow(
    u_matrix,
    origin="upper"
)

ax.set_title(
    "SOM U-Matrix — Fold 3, 13×13"
)

ax.set_xlabel("SOM column")
ax.set_ylabel("SOM row")

fig.colorbar(
    im,
    ax=ax,
    label="Mean neighbor distance"
)

plt.tight_layout()

som_umatrix_output = (
    FIGURE_DIR /
    "Paper1_SOM_Fold3_13x13_Umatrix.png"
)

plt.savefig(
    som_umatrix_output,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\nU-Matrix saved to:")
print(som_umatrix_output)

print("\nCell 15 completed.")

In [ ]:
# ============================================================
# Cell 16: SOM domain-number sweep
#          Cluster SOM neuron prototypes for k = 2 ... 10
#          Fold 3, 13x13
# ============================================================

# ------------------------------------------------------------
# Retrieve SOM codebook / neuron weight vectors
#
# shape:
# (SOM_X, SOM_Y, PCA_features)
# ------------------------------------------------------------

som_weights = som.get_weights()

print("SOM weight tensor shape:")
print(som_weights.shape)

# Flatten 5x5 neurons to 25 prototype vectors
som_prototypes = som_weights.reshape(
    SOM_X * SOM_Y,
    -1
)

print(
    "Flattened SOM prototypes:",
    som_prototypes.shape
)

# ------------------------------------------------------------
# Convert BMU coordinates to flat neuron indices
# ------------------------------------------------------------

def bmu_to_index(bmu_array, som_y):

    return (
        bmu_array[:, 0] * som_y
        + bmu_array[:, 1]
    ).astype(int)


train_bmu_index = bmu_to_index(
    train_bmu,
    SOM_Y
)

val_bmu_index = bmu_to_index(
    val_bmu,
    SOM_Y
)

# ------------------------------------------------------------
# Sweep domain number k = 2 ... 10
# ------------------------------------------------------------

SOM_K_VALUES = list(range(2, 11))

som_k_records = []
som_k_models = {}

for k in SOM_K_VALUES:

    # --------------------------------------------------------
    # Cluster SOM neuron prototypes
    # --------------------------------------------------------

    neuron_kmeans = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=50,
        algorithm="lloyd"
    )

    neuron_domain_labels = (
        neuron_kmeans.fit_predict(
            som_prototypes
        )
    )

    # --------------------------------------------------------
    # Map sample BMUs to SOM-domain labels
    # --------------------------------------------------------

    train_domain_labels = (
        neuron_domain_labels[
            train_bmu_index
        ]
    )

    val_domain_labels = (
        neuron_domain_labels[
            val_bmu_index
        ]
    )

    # --------------------------------------------------------
    # Make sure TRAIN uses at least 2 domains
    # --------------------------------------------------------

    n_train_domains = len(
        np.unique(train_domain_labels)
    )

    if n_train_domains < 2:

        silhouette = np.nan
        dbi = np.nan
        ch = np.nan

    else:

        # Evaluate on actual PCA sample representations
        silhouette = silhouette_score(
            Z_train_som,
            train_domain_labels
        )

        dbi = davies_bouldin_score(
            Z_train_som,
            train_domain_labels
        )

        ch = calinski_harabasz_score(
            Z_train_som,
            train_domain_labels
        )

    # --------------------------------------------------------
    # Sample occupancy
    # --------------------------------------------------------

    train_counts = np.bincount(
        train_domain_labels,
        minlength=k
    )

    val_counts = np.bincount(
        val_domain_labels,
        minlength=k
    )

    # --------------------------------------------------------
    # Neuron occupancy per domain
    # --------------------------------------------------------

    neuron_counts = np.bincount(
        neuron_domain_labels,
        minlength=k
    )

    som_k_records.append({
        "k": k,

        "Silhouette": silhouette,
        "Davies_Bouldin": dbi,
        "Calinski_Harabasz": ch,

        "TRAIN_min_cluster":
            int(train_counts.min()),

        "TRAIN_max_cluster":
            int(train_counts.max()),

        "VALIDATION_min_cluster":
            int(val_counts.min()),

        "VALIDATION_max_cluster":
            int(val_counts.max()),

        "Empty_validation_clusters":
            int(np.sum(val_counts == 0)),

        "Min_neurons_per_domain":
            int(neuron_counts.min()),

        "Max_neurons_per_domain":
            int(neuron_counts.max())
    })

    som_k_models[k] = {
        "model": neuron_kmeans,
        "neuron_labels":
            neuron_domain_labels,
        "train_labels":
            train_domain_labels,
        "val_labels":
            val_domain_labels
    }

    print(
        f"k={k:2d} | "
        f"Sil={silhouette:.3f} | "
        f"DB={dbi:.3f} | "
        f"CH={ch:.1f} | "
        f"TRAIN min={train_counts.min():3d} | "
        f"VAL min={val_counts.min():3d} | "
        f"Empty VAL={np.sum(val_counts == 0)}"
    )

# ------------------------------------------------------------
# Results table
# ------------------------------------------------------------

som_k_sweep_df = pd.DataFrame(
    som_k_records
)

print("\nSOM DOMAIN-NUMBER SWEEP")
print("=" * 75)

display(
    som_k_sweep_df.round(4)
)

# ------------------------------------------------------------
# Create output folder
# ------------------------------------------------------------

SOM_DIR = (
    BASELINE_DIR /
    "SOM"
)

SOM_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

som_k_output = (
    SOM_DIR /
    "Fold3_13x13_SOM5x5_KMeans_k2to10.csv"
)

som_k_sweep_df.to_csv(
    som_k_output,
    index=False
)

print("\nSaved to:")
print(som_k_output)

print("\nCell 16 completed.")

In [ ]:
# ============================================================
# Cell 17: SOM stability test
#          Fold 3, 13x13
#          SOM 5x5 + k=2
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

from itertools import combinations

FINAL_SOM_K = 2
N_SOM_STABILITY_RUNS = 30
SOM_ITERATIONS = 5000

som_stability_labels = []

for seed in range(N_SOM_STABILITY_RUNS):

    # --------------------------------------------------------
    # Train independent SOM
    # --------------------------------------------------------

    som_run = MiniSom(
        x=5,
        y=5,
        input_len=Z_train_som.shape[1],
        sigma=1.0,
        learning_rate=0.5,
        neighborhood_function="gaussian",
        random_seed=seed
    )

    som_run.pca_weights_init(
        Z_train_som
    )

    som_run.train_random(
        data=Z_train_som,
        num_iteration=SOM_ITERATIONS,
        verbose=False
    )

    # --------------------------------------------------------
    # Obtain BMUs
    # --------------------------------------------------------

    train_bmu_run = np.array(
        [
            som_run.winner(x)
            for x in Z_train_som
        ]
    )

    train_bmu_index_run = (
        train_bmu_run[:, 0] * 5
        + train_bmu_run[:, 1]
    ).astype(int)

    # --------------------------------------------------------
    # Cluster SOM neuron prototypes into k=2 domains
    # --------------------------------------------------------

    prototypes_run = (
        som_run
        .get_weights()
        .reshape(
            25,
            -1
        )
    )

    neuron_model = KMeans(
        n_clusters=FINAL_SOM_K,
        random_state=RANDOM_STATE,
        n_init=50,
        algorithm="lloyd"
    )

    neuron_labels = neuron_model.fit_predict(
        prototypes_run
    )

    sample_domain_labels = (
        neuron_labels[
            train_bmu_index_run
        ]
    )

    som_stability_labels.append(
        sample_domain_labels
    )

# ------------------------------------------------------------
# Pairwise ARI / NMI
# ------------------------------------------------------------

ari_values = []
nmi_values = []

for i, j in combinations(
    range(N_SOM_STABILITY_RUNS),
    2
):

    ari_values.append(
        adjusted_rand_score(
            som_stability_labels[i],
            som_stability_labels[j]
        )
    )

    nmi_values.append(
        normalized_mutual_info_score(
            som_stability_labels[i],
            som_stability_labels[j]
        )
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

som_stability_df = pd.DataFrame(
    [{
        "SOM_grid": "5x5",
        "k": FINAL_SOM_K,
        "Runs": N_SOM_STABILITY_RUNS,

        "ARI_mean": np.mean(ari_values),
        "ARI_std": np.std(
            ari_values,
            ddof=1
        ),
        "ARI_min": np.min(ari_values),
        "ARI_median": np.median(ari_values),

        "NMI_mean": np.mean(nmi_values),
        "NMI_std": np.std(
            nmi_values,
            ddof=1
        ),
        "NMI_min": np.min(nmi_values),
        "NMI_median": np.median(nmi_values)
    }]
)

display(
    som_stability_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

som_stability_output = (
    SOM_DIR /
    "Fold3_13x13_SOM5x5_k2_stability.csv"
)

som_stability_df.to_csv(
    som_stability_output,
    index=False
)

print("\nSaved to:")
print(som_stability_output)

print("\nCell 17 completed.")

In [ ]:
# ============================================================
# Cell 18: Full five-fold SOM baseline
#          PCA(95%) -> SOM 5x5 -> neuron KMeans(k=2)
#          for 9x9 and 13x13 patches
# ============================================================

FINAL_SOM_K = 2
SOM_X = 5
SOM_Y = 5
SOM_ITERATIONS = 5000

full_som_records = []

for patch_size in PATCH_SIZES:

    print("\n" + "=" * 80)
    print(f"PATCH SIZE: {patch_size} x {patch_size}")
    print("=" * 80)

    for fold in range(1, N_FOLDS + 1):

        print(
            f"\nProcessing patch {patch_size}x{patch_size}, "
            f"Fold {fold}"
        )

        # ----------------------------------------------------
        # Load fold tensors
        # ----------------------------------------------------

        fold_dir = (
            PATCH_DIR /
            f"{patch_size}x{patch_size}" /
            f"Fold_{fold}"
        )

        train_file = (
            fold_dir /
            f"Fold{fold}_TRAIN_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        val_file = (
            fold_dir /
            f"Fold{fold}_VALIDATION_"
            f"{patch_size}x{patch_size}_14ch.npy"
        )

        X_train_patch = np.load(train_file)
        X_val_patch = np.load(val_file)

        # ----------------------------------------------------
        # Flatten
        # ----------------------------------------------------

        X_train_flat = X_train_patch.reshape(
            X_train_patch.shape[0],
            -1
        )

        X_val_flat = X_val_patch.reshape(
            X_val_patch.shape[0],
            -1
        )

        # ----------------------------------------------------
        # PCA fit on TRAIN only
        # ----------------------------------------------------

        pca = PCA(
            n_components=0.95,
            svd_solver="full",
            random_state=RANDOM_STATE
        )

        Z_train = pca.fit_transform(
            X_train_flat
        )

        Z_val = pca.transform(
            X_val_flat
        )

        # ----------------------------------------------------
        # Train SOM on TRAIN only
        # ----------------------------------------------------

        som_model = MiniSom(
            x=SOM_X,
            y=SOM_Y,
            input_len=Z_train.shape[1],
            sigma=1.0,
            learning_rate=0.5,
            neighborhood_function="gaussian",
            random_seed=RANDOM_STATE
        )

        som_model.pca_weights_init(
            Z_train
        )

        som_model.train_random(
            data=Z_train,
            num_iteration=SOM_ITERATIONS,
            verbose=False
        )

        # ----------------------------------------------------
        # Quantization errors
        # ----------------------------------------------------

        train_qe = som_model.quantization_error(
            Z_train
        )

        val_qe = som_model.quantization_error(
            Z_val
        )

        # ----------------------------------------------------
        # BMUs
        # ----------------------------------------------------

        train_bmu = np.array([
            som_model.winner(x)
            for x in Z_train
        ])

        val_bmu = np.array([
            som_model.winner(x)
            for x in Z_val
        ])

        train_bmu_idx = (
            train_bmu[:, 0] * SOM_Y
            + train_bmu[:, 1]
        ).astype(int)

        val_bmu_idx = (
            val_bmu[:, 0] * SOM_Y
            + val_bmu[:, 1]
        ).astype(int)

        # ----------------------------------------------------
        # SOM neuron prototypes
        # ----------------------------------------------------

        prototypes = (
            som_model
            .get_weights()
            .reshape(
                SOM_X * SOM_Y,
                -1
            )
        )

        # ----------------------------------------------------
        # Cluster neuron prototypes into k=2 domains
        # ----------------------------------------------------

        neuron_kmeans = KMeans(
            n_clusters=FINAL_SOM_K,
            random_state=RANDOM_STATE,
            n_init=50,
            algorithm="lloyd"
        )

        neuron_labels = neuron_kmeans.fit_predict(
            prototypes
        )

        train_labels = neuron_labels[
            train_bmu_idx
        ]

        val_labels = neuron_labels[
            val_bmu_idx
        ]

        # ----------------------------------------------------
        # TRAIN clustering metrics
        # ----------------------------------------------------

        silhouette = silhouette_score(
            Z_train,
            train_labels
        )

        dbi = davies_bouldin_score(
            Z_train,
            train_labels
        )

        ch = calinski_harabasz_score(
            Z_train,
            train_labels
        )

        # ----------------------------------------------------
        # Occupancy diagnostics
        # ----------------------------------------------------

        train_counts = np.bincount(
            train_labels,
            minlength=FINAL_SOM_K
        )

        val_counts = np.bincount(
            val_labels,
            minlength=FINAL_SOM_K
        )

        occupied_train_neurons = len(
            np.unique(
                train_bmu,
                axis=0
            )
        )

        occupied_val_neurons = len(
            np.unique(
                val_bmu,
                axis=0
            )
        )

        full_som_records.append({
            "Patch_size": patch_size,
            "Validation_Fold": fold,

            "TRAIN_patches": len(X_train_patch),
            "VALIDATION_patches": len(X_val_patch),

            "PCA_components_95":
                Z_train.shape[1],

            "Retained_variance":
                pca.explained_variance_ratio_.sum(),

            "TRAIN_quantization_error":
                train_qe,

            "VALIDATION_quantization_error":
                val_qe,

            "QE_ratio_VAL_to_TRAIN":
                val_qe / train_qe,

            "Occupied_TRAIN_neurons":
                occupied_train_neurons,

            "Occupied_VALIDATION_neurons":
                occupied_val_neurons,

            "Silhouette":
                silhouette,

            "Davies_Bouldin":
                dbi,

            "Calinski_Harabasz":
                ch,

            "TRAIN_cluster_min":
                int(train_counts.min()),

            "TRAIN_cluster_max":
                int(train_counts.max()),

            "VALIDATION_cluster_min":
                int(val_counts.min()),

            "VALIDATION_cluster_max":
                int(val_counts.max()),

            "Empty_validation_clusters":
                int(np.sum(val_counts == 0))
        })

        print(
            f"PCs={Z_train.shape[1]:3d} | "
            f"QEtr={train_qe:.2f} | "
            f"QEval={val_qe:.2f} | "
            f"Sil={silhouette:.3f} | "
            f"DB={dbi:.3f} | "
            f"VAL min={val_counts.min()}"
        )

# ------------------------------------------------------------
# Full results table
# ------------------------------------------------------------

full_som_df = pd.DataFrame(
    full_som_records
)

print("\n" + "=" * 80)
print("FULL SOM BASELINE RESULTS")
print("=" * 80)

display(
    full_som_df.round(4)
)

# ------------------------------------------------------------
# Patch-size summary
# ------------------------------------------------------------

som_patch_summary = (
    full_som_df
    .groupby("Patch_size")
    .agg(
        mean_train_QE=(
            "TRAIN_quantization_error",
            "mean"
        ),
        mean_validation_QE=(
            "VALIDATION_quantization_error",
            "mean"
        ),
        mean_QE_ratio=(
            "QE_ratio_VAL_to_TRAIN",
            "mean"
        ),
        mean_Silhouette=(
            "Silhouette",
            "mean"
        ),
        std_Silhouette=(
            "Silhouette",
            "std"
        ),
        mean_Davies_Bouldin=(
            "Davies_Bouldin",
            "mean"
        ),
        mean_Calinski_Harabasz=(
            "Calinski_Harabasz",
            "mean"
        ),
        min_validation_cluster=(
            "VALIDATION_cluster_min",
            "min"
        ),
        mean_occupied_validation_neurons=(
            "Occupied_VALIDATION_neurons",
            "mean"
        )
    )
    .reset_index()
)

print("\nPATCH-SIZE SUMMARY")

display(
    som_patch_summary.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

full_som_output = (
    SOM_DIR /
    "Paper1_SOM5x5_k2_full_5fold_results.csv"
)

som_summary_output = (
    SOM_DIR /
    "Paper1_SOM5x5_k2_patchsize_summary.csv"
)

full_som_df.to_csv(
    full_som_output,
    index=False
)

som_patch_summary.to_csv(
    som_summary_output,
    index=False
)

print("\nSaved:")
print(full_som_output)
print(som_summary_output)

print("\nCell 18 completed.")

In [ ]:
# ============================================================
# Cell 19: Consolidated comparison of completed baselines
#          PCA + KMeans
#          NMF + KMeans
#          SOM
# ============================================================

# ------------------------------------------------------------
# Load saved patch-size summaries
# ------------------------------------------------------------

pca_summary_file = (
    PCA_DIR /
    "Paper1_PCA95_KMeans_k3_patchsize_summary.csv"
)

nmf_summary_file = (
    NMF_DIR /
    "Paper1_NMF10_KMeans_k2_patchsize_summary.csv"
)

som_summary_file = (
    SOM_DIR /
    "Paper1_SOM5x5_k2_patchsize_summary.csv"
)

pca_summary = pd.read_csv(
    pca_summary_file
)

nmf_summary = pd.read_csv(
    nmf_summary_file
)

som_summary = pd.read_csv(
    som_summary_file
)

# ------------------------------------------------------------
# Build standardized comparison table
# ------------------------------------------------------------

comparison_records = []

for _, row in pca_summary.iterrows():

    comparison_records.append({
        "Baseline": "PCA + KMeans",
        "Patch_size": int(row["Patch_size"]),
        "Latent_configuration": "PCA 95% variance",
        "Domain_k": 3,

        "Mean_Silhouette":
            row["mean_Silhouette"],

        "Mean_Davies_Bouldin":
            row["mean_Davies_Bouldin"],

        "Mean_Calinski_Harabasz":
            row["mean_Calinski_Harabasz"],

        "Min_validation_cluster":
            row["min_validation_cluster"],

        "Extra_metric_name":
            "Mean PCA components",

        "Extra_metric_value":
            row["mean_PCs"]
    })


for _, row in nmf_summary.iterrows():

    comparison_records.append({
        "Baseline": "NMF + KMeans",
        "Patch_size": int(row["Patch_size"]),
        "Latent_configuration": "NMF rank 10",
        "Domain_k": 2,

        "Mean_Silhouette":
            row["mean_Silhouette"],

        "Mean_Davies_Bouldin":
            row["mean_Davies_Bouldin"],

        "Mean_Calinski_Harabasz":
            row["mean_Calinski_Harabasz"],

        "Min_validation_cluster":
            row["min_validation_cluster"],

        "Extra_metric_name":
            "Mean reconstruction error",

        "Extra_metric_value":
            row["mean_reconstruction_error"]
    })


for _, row in som_summary.iterrows():

    comparison_records.append({
        "Baseline": "SOM",
        "Patch_size": int(row["Patch_size"]),
        "Latent_configuration": "PCA95 -> SOM 5x5",
        "Domain_k": 2,

        "Mean_Silhouette":
            row["mean_Silhouette"],

        "Mean_Davies_Bouldin":
            row["mean_Davies_Bouldin"],

        "Mean_Calinski_Harabasz":
            row["mean_Calinski_Harabasz"],

        "Min_validation_cluster":
            row["min_validation_cluster"],

        "Extra_metric_name":
            "Mean VAL/TRAIN QE",

        "Extra_metric_value":
            row["mean_QE_ratio"]
    })


baseline_comparison_df = pd.DataFrame(
    comparison_records
)

# ------------------------------------------------------------
# Sort for readability
# ------------------------------------------------------------

baseline_comparison_df = (
    baseline_comparison_df
    .sort_values(
        [
            "Patch_size",
            "Baseline"
        ]
    )
    .reset_index(drop=True)
)

print("COMPLETED BASELINE COMPARISON")
print("=" * 80)

display(
    baseline_comparison_df.round(4)
)

# ------------------------------------------------------------
# Add simple within-patch rankings
# ------------------------------------------------------------

ranked_df = baseline_comparison_df.copy()

ranked_df[
    "Silhouette_rank"
] = (
    ranked_df
    .groupby("Patch_size")[
        "Mean_Silhouette"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

ranked_df[
    "DBI_rank"
] = (
    ranked_df
    .groupby("Patch_size")[
        "Mean_Davies_Bouldin"
    ]
    .rank(
        ascending=True,
        method="min"
    )
)

ranked_df[
    "CH_rank"
] = (
    ranked_df
    .groupby("Patch_size")[
        "Mean_Calinski_Harabasz"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)

ranked_df[
    "Mean_internal_rank"
] = (
    ranked_df[
        [
            "Silhouette_rank",
            "DBI_rank",
            "CH_rank"
        ]
    ]
    .mean(axis=1)
)

print("\nWITHIN-PATCH INTERNAL-METRIC RANKING")
print("=" * 80)

display(
    ranked_df[
        [
            "Baseline",
            "Patch_size",
            "Domain_k",
            "Mean_Silhouette",
            "Mean_Davies_Bouldin",
            "Mean_Calinski_Harabasz",
            "Min_validation_cluster",
            "Mean_internal_rank"
        ]
    ]
    .sort_values(
        [
            "Patch_size",
            "Mean_internal_rank"
        ]
    )
    .round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

comparison_output = (
    BASELINE_DIR /
    "Paper1_completed_baseline_comparison.csv"
)

ranked_output = (
    BASELINE_DIR /
    "Paper1_completed_baseline_rankings.csv"
)

baseline_comparison_df.to_csv(
    comparison_output,
    index=False
)

ranked_df.to_csv(
    ranked_output,
    index=False
)

print("\nSaved:")
print(comparison_output)
print(ranked_output)

print("\nCell 19 completed.")

In [ ]:
# ============================================================
# Cell 20: Check PyTorch installation and GPU availability
# ============================================================

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim

    print("PyTorch version:", torch.__version__)
    print("CUDA available :", torch.cuda.is_available())

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print(
            "CUDA version:",
            torch.version.cuda
        )

except ImportError:

    print("PyTorch is not installed in paper1_ssl.")

In [ ]:
# ============================================================
# Cell 21: Define Convolutional Autoencoder baseline
#          Input: (N, 14, H, W)
#          Latent dimension: 32
# ============================================================

import torch
import torch.nn as nn

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

LATENT_DIM = 32

print("Using device:", DEVICE)


class ConvAutoencoder(nn.Module):

    def __init__(
        self,
        in_channels=14,
        latent_dim=32
    ):
        super().__init__()

        # ----------------------------------------------------
        # Encoder
        # ----------------------------------------------------
        self.encoder_conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(
                (3, 3)
            )
        )

        # 64 channels × 3 × 3
        self.encoder_fc = nn.Linear(
            64 * 3 * 3,
            latent_dim
        )

        # ----------------------------------------------------
        # Decoder
        #
        # Output spatial dimensions will be matched explicitly
        # to the original patch size during forward pass.
        # ----------------------------------------------------
        self.decoder_fc = nn.Linear(
            latent_dim,
            64 * 3 * 3
        )

        self.decoder_conv = nn.Sequential(

            nn.Conv2d(
                64,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                in_channels,
                kernel_size=3,
                padding=1
            )
        )

    def encode(self, x):

        x = self.encoder_conv(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        z = self.encoder_fc(x)

        return z

    def decode(
        self,
        z,
        output_size
    ):

        x = self.decoder_fc(z)

        x = x.view(
            -1,
            64,
            3,
            3
        )

        # Upsample back to original patch dimensions
        x = nn.functional.interpolate(
            x,
            size=output_size,
            mode="bilinear",
            align_corners=False
        )

        x = self.decoder_conv(x)

        return x

    def forward(self, x):

        output_size = (
            x.shape[-2],
            x.shape[-1]
        )

        z = self.encode(x)

        recon = self.decode(
            z,
            output_size
        )

        return recon, z


# ------------------------------------------------------------
# Fixed configuration
# ------------------------------------------------------------

N_CHANNELS = 14
LATENT_DIM = 32

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", DEVICE)

# ------------------------------------------------------------
# Instantiate model
# ------------------------------------------------------------

cae_test = ConvAutoencoder(
    in_channels=N_CHANNELS,
    latent_dim=LATENT_DIM
).to(DEVICE)

print(cae_test)

# ------------------------------------------------------------
# Parameter count
# ------------------------------------------------------------

total_params = sum(
    p.numel()
    for p in cae_test.parameters()
)

trainable_params = sum(
    p.numel()
    for p in cae_test.parameters()
    if p.requires_grad
)

print(
    "\nTotal parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

print("\nCell 21 completed.")

In [ ]:
# ============================================================
# Cell 22: Forward-pass sanity test
#          Fold 3, 13x13
# ============================================================

from pathlib import Path
import numpy as np

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

PATCH_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "patches"
)

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

# ------------------------------------------------------------
# Locate Fold 3, 13x13 tensors
# ------------------------------------------------------------

fold_dir = (
    PATCH_DIR /
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}" /
    f"Fold_{TEST_FOLD}"
)

train_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_TRAIN_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

val_file = (
    fold_dir /
    f"Fold{TEST_FOLD}_VALIDATION_"
    f"{TEST_PATCH_SIZE}x{TEST_PATCH_SIZE}_14ch.npy"
)

print("TRAIN file exists:", train_file.exists())
print("VAL file exists  :", val_file.exists())

print("\nTRAIN path:")
print(train_file)

print("\nVAL path:")
print(val_file)

# ------------------------------------------------------------
# Load tensors
# ------------------------------------------------------------

X_train_cae = np.load(
    train_file
).astype(np.float32)

X_val_cae = np.load(
    val_file
).astype(np.float32)

print("\nOriginal TRAIN shape:", X_train_cae.shape)
print("Original VAL shape  :", X_val_cae.shape)

# ------------------------------------------------------------
# Convert channels-last numpy arrays:
# (N, H, W, C)
#
# to PyTorch channels-first:
# (N, C, H, W)
# ------------------------------------------------------------

X_train_cae_t = torch.from_numpy(
    np.transpose(
        X_train_cae,
        (0, 3, 1, 2)
    )
)

X_val_cae_t = torch.from_numpy(
    np.transpose(
        X_val_cae,
        (0, 3, 1, 2)
    )
)

print("\nPyTorch TRAIN shape:", X_train_cae_t.shape)
print("PyTorch VAL shape  :", X_val_cae_t.shape)

# ------------------------------------------------------------
# Finite-value checks
# ------------------------------------------------------------

print(
    "\nTRAIN finite:",
    torch.isfinite(
        X_train_cae_t
    ).all().item()
)

print(
    "VAL finite:",
    torch.isfinite(
        X_val_cae_t
    ).all().item()
)

# ------------------------------------------------------------
# Small GPU batch
# ------------------------------------------------------------

test_batch = (
    X_train_cae_t[:16]
    .to(DEVICE)
)

print(
    "\nTest batch shape:",
    test_batch.shape
)

# ------------------------------------------------------------
# Forward pass
# ------------------------------------------------------------

cae_test.eval()

with torch.no_grad():

    recon_test, latent_test = cae_test(
        test_batch
    )

print(
    "Reconstruction shape:",
    recon_test.shape
)

print(
    "Latent shape:",
    latent_test.shape
)

# ------------------------------------------------------------
# Initial reconstruction loss
# ------------------------------------------------------------

initial_mse = (
    nn.functional.mse_loss(
        recon_test,
        test_batch
    )
    .item()
)

print(
    "\nInitial untrained MSE:",
    round(initial_mse, 6)
)

# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert (
    recon_test.shape ==
    test_batch.shape
), "Reconstruction shape does not match input."

assert (
    latent_test.shape[1] ==
    LATENT_DIM
), "Latent dimension is incorrect."

assert torch.isfinite(
    recon_test
).all(), \
    "Non-finite reconstruction values detected."

assert torch.isfinite(
    latent_test
).all(), \
    "Non-finite latent values detected."

print("\nAll forward-pass checks PASSED.")
print("Cell 22 completed.")

In [ ]:
# ============================================================
# Cell 23: Stable CAE training + checkpointing
#          Fold 3, 13x13
#          CPU execution for Windows stability
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
import time

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"C:\project 3\Paper1_SSL")

PATCH_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "patches"
)

CAE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "baselines" /
    "CAE"
)

CAE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

TEST_PATCH_SIZE = 13
TEST_FOLD = 3

N_CHANNELS = 14
LATENT_DIM = 32

CAE_SEED = 42
CAE_EPOCHS = 200
CAE_BATCH_SIZE = 32
CAE_LR = 1e-3
CAE_WEIGHT_DECAY = 1e-5

# Deliberately use CPU for this small baseline
# to avoid the Windows CUDA/DLL instability.
DEVICE = torch.device("cpu")

print("Training device:", DEVICE)


# ------------------------------------------------------------
# CAE architecture
# ------------------------------------------------------------

class ConvAutoencoder(nn.Module):

    def __init__(
        self,
        in_channels=14,
        latent_dim=32
    ):
        super().__init__()

        self.encoder_conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d((3, 3))
        )

        self.encoder_fc = nn.Linear(
            64 * 3 * 3,
            latent_dim
        )

        self.decoder_fc = nn.Linear(
            latent_dim,
            64 * 3 * 3
        )

        self.decoder_conv = nn.Sequential(

            nn.Conv2d(
                64,
                32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(inplace=True),

            nn.Conv2d(
                32,
                in_channels,
                kernel_size=3,
                padding=1
            )
        )

    def encode(self, x):

        x = self.encoder_conv(x)

        x = torch.flatten(
            x,
            start_dim=1
        )

        return self.encoder_fc(x)

    def decode(
        self,
        z,
        output_size
    ):

        x = self.decoder_fc(z)

        x = x.view(
            -1,
            64,
            3,
            3
        )

        x = nn.functional.interpolate(
            x,
            size=output_size,
            mode="bilinear",
            align_corners=False
        )

        return self.decoder_conv(x)

    def forward(self, x):

        output_size = (
            x.shape[-2],
            x.shape[-1]
        )

        z = self.encode(x)

        recon = self.decode(
            z,
            output_size
        )

        return recon, z


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

random.seed(CAE_SEED)
np.random.seed(CAE_SEED)
torch.manual_seed(CAE_SEED)

torch.set_num_threads(2)

# ------------------------------------------------------------
# Load Fold 3 TRAIN patches
# ------------------------------------------------------------

fold_dir = (
    PATCH_DIR /
    "13x13" /
    "Fold_3"
)

train_file = (
    fold_dir /
    "Fold3_TRAIN_13x13_14ch.npy"
)

X_train = np.load(
    train_file
).astype(np.float32)

X_train_t = torch.from_numpy(
    np.transpose(
        X_train,
        (0, 3, 1, 2)
    )
)

print("TRAIN tensor:", X_train_t.shape)


# ------------------------------------------------------------
# DataLoader
# ------------------------------------------------------------

dataset = TensorDataset(
    X_train_t
)

generator = torch.Generator()
generator.manual_seed(CAE_SEED)

loader = DataLoader(
    dataset,
    batch_size=CAE_BATCH_SIZE,
    shuffle=True,
    generator=generator,

    # Keep loader simple/stable on Windows
    num_workers=0,
    pin_memory=False
)


# ------------------------------------------------------------
# Model / optimizer
# ------------------------------------------------------------

cae_model = ConvAutoencoder(
    in_channels=N_CHANNELS,
    latent_dim=LATENT_DIM
).to(DEVICE)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    cae_model.parameters(),
    lr=CAE_LR,
    weight_decay=CAE_WEIGHT_DECAY
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

train_loss_history = []

start_time = time.time()

for epoch in range(1, CAE_EPOCHS + 1):

    cae_model.train()

    running_loss = 0.0
    n_samples = 0

    for (batch_x,) in loader:

        batch_x = batch_x.to(DEVICE)

        optimizer.zero_grad(
            set_to_none=True
        )

        recon, _ = cae_model(
            batch_x
        )

        loss = criterion(
            recon,
            batch_x
        )

        loss.backward()

        optimizer.step()

        batch_n = batch_x.size(0)

        running_loss += (
            loss.item() *
            batch_n
        )

        n_samples += batch_n

    epoch_loss = (
        running_loss /
        n_samples
    )

    train_loss_history.append(
        epoch_loss
    )

    # --------------------------------------------------------
    # Every 20 epochs:
    # print + checkpoint model/history
    # --------------------------------------------------------

    if (
        epoch == 1
        or epoch % 20 == 0
        or epoch == CAE_EPOCHS
    ):

        print(
            f"Epoch {epoch:3d}/{CAE_EPOCHS} | "
            f"TRAIN MSE = {epoch_loss:.6f}"
        )

        checkpoint_file = (
            CAE_DIR /
            f"Fold3_13x13_CAE_checkpoint_epoch{epoch:03d}.pt"
        )

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict":
                    cae_model.state_dict(),
                "optimizer_state_dict":
                    optimizer.state_dict(),
                "train_loss_history":
                    train_loss_history,
                "latent_dim":
                    LATENT_DIM,
                "seed":
                    CAE_SEED
            },
            checkpoint_file
        )


elapsed = time.time() - start_time

print(
    "\nTraining time:",
    round(elapsed, 2),
    "seconds"
)


# ------------------------------------------------------------
# Final full-TRAIN reconstruction MSE
# ------------------------------------------------------------

cae_model.eval()

total_loss = 0.0
total_n = 0

with torch.no_grad():

    for (batch_x,) in loader:

        batch_x = batch_x.to(DEVICE)

        recon, _ = cae_model(
            batch_x
        )

        loss = criterion(
            recon,
            batch_x
        )

        batch_n = batch_x.size(0)

        total_loss += (
            loss.item() *
            batch_n
        )

        total_n += batch_n


final_train_mse = (
    total_loss /
    total_n
)

print(
    "\nFinal full-TRAIN MSE:",
    round(final_train_mse, 6)
)


# ------------------------------------------------------------
# Save final model
# ------------------------------------------------------------

final_model_file = (
    CAE_DIR /
    "Fold3_13x13_CAE_latent32_FINAL.pt"
)

torch.save(
    {
        "model_state_dict":
            cae_model.state_dict(),

        "patch_size":
            TEST_PATCH_SIZE,

        "fold":
            TEST_FOLD,

        "latent_dim":
            LATENT_DIM,

        "n_channels":
            N_CHANNELS,

        "epochs":
            CAE_EPOCHS,

        "learning_rate":
            CAE_LR,

        "weight_decay":
            CAE_WEIGHT_DECAY,

        "seed":
            CAE_SEED,

        "final_train_mse":
            final_train_mse
    },
    final_model_file
)


# ------------------------------------------------------------
# Save loss history
# ------------------------------------------------------------

history_df = pd.DataFrame({

    "Epoch":
        np.arange(
            1,
            len(train_loss_history) + 1
        ),

    "TRAIN_MSE":
        train_loss_history
})

history_file = (
    CAE_DIR /
    "Fold3_13x13_CAE_training_history.csv"
)

history_df.to_csv(
    history_file,
    index=False
)


# ------------------------------------------------------------
# Save convergence figure
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    history_df["Epoch"],
    history_df["TRAIN_MSE"]
)

plt.xlabel("Epoch")
plt.ylabel("Training reconstruction MSE")

plt.title(
    "CAE Training Convergence — Fold 3, 13×13"
)

plt.grid(alpha=0.25)

plt.tight_layout()

plot_file = (
    CAE_DIR /
    "Fold3_13x13_CAE_training_curve.png"
)

plt.savefig(
    plot_file,
    dpi=300,
    bbox_inches="tight"
)

plt.show()


print("\nFINAL model saved:")
print(final_model_file)

print("\nHistory saved:")
print(history_file)

print("\nFigure saved:")
print(plot_file)

print("\nCell 23 completed successfully.")

In [ ]:
# ============================================================
# Cell 24: CAE latent-space KMeans sweep
#          Fold 3, 13x13
#          k = 2 ... 10
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

PROJECT_ROOT = Path(
    r"C:\project 3\Paper1_SSL"
)

CAE_DIR = (
    PROJECT_ROOT /
    "03_outputs" /
    "baselines" /
    "CAE"
)

train_latent_file = (
    CAE_DIR /
    "Fold3_13x13_CAE_TRAIN_latent32.npy"
)

val_latent_file = (
    CAE_DIR /
    "Fold3_13x13_CAE_VALIDATION_latent32.npy"
)

Z_train_cae = np.load(
    train_latent_file
)

Z_val_cae = np.load(
    val_latent_file
)

print("TRAIN latent:", Z_train_cae.shape)
print("VALIDATION latent:", Z_val_cae.shape)

K_VALUES = range(2, 11)

records = []

for k in K_VALUES:

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50,
        algorithm="lloyd"
    )

    train_labels = model.fit_predict(
        Z_train_cae
    )

    val_labels = model.predict(
        Z_val_cae
    )

    silhouette = silhouette_score(
        Z_train_cae,
        train_labels
    )

    dbi = davies_bouldin_score(
        Z_train_cae,
        train_labels
    )

    ch = calinski_harabasz_score(
        Z_train_cae,
        train_labels
    )

    train_counts = np.bincount(
        train_labels,
        minlength=k
    )

    val_counts = np.bincount(
        val_labels,
        minlength=k
    )

    records.append({
        "k": k,
        "Silhouette": silhouette,
        "Davies_Bouldin": dbi,
        "Calinski_Harabasz": ch,
        "TRAIN_min_cluster":
            int(train_counts.min()),
        "TRAIN_max_cluster":
            int(train_counts.max()),
        "VALIDATION_min_cluster":
            int(val_counts.min()),
        "VALIDATION_max_cluster":
            int(val_counts.max()),
        "Empty_validation_clusters":
            int(np.sum(val_counts == 0))
    })

    print(
        f"k={k:2d} | "
        f"Sil={silhouette:.4f} | "
        f"DB={dbi:.4f} | "
        f"CH={ch:.2f} | "
        f"TRAIN min={train_counts.min():3d} | "
        f"VAL min={val_counts.min():3d} | "
        f"Empty VAL={np.sum(val_counts == 0)}"
    )

cae_k_sweep_df = pd.DataFrame(
    records
)

print("\nCAE KMEANS SWEEP")
display(
    cae_k_sweep_df.round(4)
)

output_file = (
    CAE_DIR /
    "Fold3_13x13_CAE32_KMeans_k2to10.csv"
)

cae_k_sweep_df.to_csv(
    output_file,
    index=False
)

print("\nSaved to:")
print(output_file)

print("\nCell 24 completed.")

In [ ]:
# ============================================================
# Cell 25: CAE + KMeans k=2 initialization stability
#          Fold 3, 13x13
# ============================================================

from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score
)

from itertools import combinations

FINAL_CAE_K = 2
N_STABILITY_RUNS = 50

cae_stability_labels = []

for seed in range(
    N_STABILITY_RUNS
):

    km = KMeans(
        n_clusters=FINAL_CAE_K,
        random_state=seed,
        n_init=1,
        algorithm="lloyd"
    )

    labels = km.fit_predict(
        Z_train_cae
    )

    cae_stability_labels.append(
        labels
    )

# ------------------------------------------------------------
# Pairwise ARI / NMI
# ------------------------------------------------------------

ari_values = []
nmi_values = []

for i, j in combinations(
    range(N_STABILITY_RUNS),
    2
):

    ari_values.append(
        adjusted_rand_score(
            cae_stability_labels[i],
            cae_stability_labels[j]
        )
    )

    nmi_values.append(
        normalized_mutual_info_score(
            cae_stability_labels[i],
            cae_stability_labels[j]
        )
    )

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------

cae_stability_df = pd.DataFrame(
    [{
        "Latent_dim": 32,
        "k": FINAL_CAE_K,
        "Runs": N_STABILITY_RUNS,

        "ARI_mean":
            np.mean(ari_values),

        "ARI_std":
            np.std(
                ari_values,
                ddof=1
            ),

        "ARI_min":
            np.min(ari_values),

        "ARI_median":
            np.median(ari_values),

        "NMI_mean":
            np.mean(nmi_values),

        "NMI_std":
            np.std(
                nmi_values,
                ddof=1
            ),

        "NMI_min":
            np.min(nmi_values),

        "NMI_median":
            np.median(nmi_values)
    }]
)

display(
    cae_stability_df.round(4)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

stability_file = (
    CAE_DIR /
    "Fold3_13x13_CAE32_KMeans_k2_stability.csv"
)

cae_stability_df.to_csv(
    stability_file,
    index=False
)

print("\nSaved to:")
print(stability_file)

print("\nCell 25 completed.")